# Identificación de Clusters de Países

## Clustering Jerárquico y PCA

## Planteamiento del problema
Supongamos que, tras un proyecto reciente que incluyó muchas campañas de sensibilización y programas de financiamiento, lograron recaudar alrededor de $10 millones de dólares. Ahora la directora o director ejecutivo de una ONG necesita decidir cómo usar este dinero de forma estratégica y efectiva. Los principales problemas al tomar esta decisión tienen que ver, sobre todo, con elegir a los países que más necesitan ayuda.

Y aquí es donde entras tú como analista de datos. Tu trabajo es categorizar a los países usando algunos factores socioeconómicos y de salud que determinan el desarrollo general del país. Después debes sugerir a los países en los que la dirección ejecutiva debería enfocarse más.

## Datos
El conjunto de datos contiene esos factores socioeconómicos.

In [ ]:
# Suprimir warnings
import warnings
warnings.filterwarnings('ignore')

# Importando librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# visualización
from matplotlib.pyplot import xticks
%matplotlib inline

# Personalización de la presentación de los datos
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 50)

# Para realizar clustering jerárquico
from scipy.cluster.hierarchy import linkage
from scipy.cluster.hierarchy import dendrogram
from scipy.cluster.hierarchy import cut_tree

# Importando el StandardScaler()
from sklearn.preprocessing import StandardScaler

# Importando el módulo de PCA
from sklearn.decomposition import PCA

# Para realizar clustering con KMeans
from sklearn.cluster import KMeans

## Preparación de los datos

### Carga de datos

* country: nombre del país
* child_mort: muertes de niños menores de 5 años por cada 1000 nacidos vivos
* exports: exportaciones de bienes y servicios, como % del PIB total
* health: gasto total en salud, como % del PIB total
* imports: importaciones de bienes y servicios, como % del PIB total
* income: ingreso neto por persona
* inflation: medición de la tasa de crecimiento anual del índice de precios
* life_expec: número promedio de años que viviría un recién nacido si los patrones de mortalidad actuales se mantuvieran iguales
* total_fer: número de hijos que tendría cada mujer si las tasas de fecundidad por edad actuales se mantuvieran iguales
* gdpp: el PIB per cápita, calculado como el PIB total dividido entre la población total

In [ ]:
#
data = pd.DataFrame(pd.read_csv('Country-data.csv'))

data.head(5)

In [ ]:
#revisando duplicados

sum(data.duplicated(subset = 'country')) == 0

# No hay valores duplicados

### Inspección de los datos

In [ ]:
#
data.shape

In [ ]:
#
data.info()

In [ ]:
#
data.describe().T

### Limpieza de datos

In [ ]:
#
data.isnull().sum()

In [ ]:
# No se observan valores nulos.

## Análisis exploratorio de datos

### Análisis univariado

####  Necesitamos elegir a los países que más necesitan ayuda. Por eso debemos identificar esos países usando algunos factores socioeconómicos y de salud que determinan el desarrollo general del país.

In [ ]:
# Veamos los 10 países más extremos para cada factor.

fig, axs = plt.subplots(3 , 3,figsize = (15,15))

# Mortalidad infantil: muertes de niños menores de 5 años por cada 1000 nacidos vivos

top10_child_mort = data[['country','child_mort']].sort_values('child_mort', ascending = False).head(10)
plt1 = sns.barplot(x='country', y='child_mort', data= top10_child_mort, ax = axs[0,0])
plt1.set(xlabel = '', ylabel= 'Mortalidad infantil')

# Tasa de fecundidad: número de hijos que tendría cada mujer si las tasas de fecundidad por edad actuales se mantuvieran iguales

top10_total_fer = data[['country','total_fer']].sort_values('total_fer', ascending = False).head(10)
plt1 = sns.barplot(x='country', y='total_fer', data= top10_total_fer, ax = axs[0,1])
plt1.set(xlabel = '', ylabel= 'Tasa de fecundidad')

# Esperanza de vida: número promedio de años que viviría un recién nacido si los patrones de mortalidad actuales se mantuvieran iguales

bottom10_life_expec = data[['country','life_expec']].sort_values('life_expec', ascending = True).head(10)
plt1 = sns.barplot(x='country', y='life_expec', data= bottom10_life_expec, ax = axs[0,2])
plt1.set(xlabel = '', ylabel= 'Esperanza de vida')

# Salud: gasto total en salud, como % del PIB total

bottom10_health = data[['country','health']].sort_values('health', ascending = True).head(10)
plt1 = sns.barplot(x='country', y='health', data= bottom10_health, ax = axs[1,0])
plt1.set(xlabel = '', ylabel= 'Salud')

# PIB per cápita: calculado como el PIB total dividido entre la población total

bottom10_gdpp = data[['country','gdpp']].sort_values('gdpp', ascending = True).head(10)
plt1 = sns.barplot(x='country', y='gdpp', data= bottom10_gdpp, ax = axs[1,1])
plt1.set(xlabel = '', ylabel= 'PIB per cápita')

# Ingreso per cápita: ingreso neto por persona

bottom10_income = data[['country','income']].sort_values('income', ascending = True).head(10)
plt1 = sns.barplot(x='country', y='income', data= bottom10_income, ax = axs[1,2])
plt1.set(xlabel = '', ylabel= 'Ingreso per cápita')

# Inflación: medición de la tasa de crecimiento anual del PIB total

top10_inflation = data[['country','inflation']].sort_values('inflation', ascending = False).head(10)
plt1 = sns.barplot(x='country', y='inflation', data= top10_inflation, ax = axs[2,0])
plt1.set(xlabel = '', ylabel= 'Inflación')

# Exportaciones: exportaciones de bienes y servicios, como % del PIB total

bottom10_exports = data[['country','exports']].sort_values('exports', ascending = True).head(10)
plt1 = sns.barplot(x='country', y='exports', data= bottom10_exports, ax = axs[2,1])
plt1.set(xlabel = '', ylabel= 'Exportaciones')

# Importaciones: importaciones de bienes y servicios, como % del PIB total

bottom10_imports = data[['country','imports']].sort_values('imports', ascending = True).head(10)
plt1 = sns.barplot(x='country', y='imports', data= bottom10_imports, ax = axs[2,2])
plt1.set(xlabel = '', ylabel= 'Importaciones')

for ax in fig.axes:
    plt.sca(ax)
    plt.xticks(rotation = 90)

plt.tight_layout()
plt.savefig('eda')
plt.show()


In [ ]:
# Revisemos los coeficientes de correlación para ver qué variables están altamente correlacionadas

plt.figure(figsize = (16, 10))
sns.heatmap(data.drop('country', axis=1).corr(), annot = True, cmap="YlGnBu")
plt.savefig('corrplot')
plt.show()

In [ ]:
# Podemos ver que hay alta correlación entre algunas variables; usaremos PCA para resolver este problema.

## Análisis de valores atípicos (outliers)

In [ ]:
# Veremos cómo se distribuyen los valores de cada columna usando boxplots

fig, axs = plt.subplots(3,3, figsize = (15,7.5))
plt1 = sns.boxplot(data['child_mort'], ax = axs[0,0])
plt2 = sns.boxplot(data['health'], ax = axs[0,1])
plt3 = sns.boxplot(data['life_expec'], ax = axs[0,2])
plt4 = sns.boxplot(data['total_fer'], ax = axs[1,0])
plt5 = sns.boxplot(data['income'], ax = axs[1,1])
plt6 = sns.boxplot(data['inflation'], ax = axs[1,2])
plt7 = sns.boxplot(data['gdpp'], ax = axs[2,0])
plt8 = sns.boxplot(data['imports'], ax = axs[2,1])
plt9 = sns.boxplot(data['exports'], ax = axs[2,2])

plt.tight_layout()

In [ ]:
# Generar un pairplot con KDE en la diagonal y scatter plots para los demás pares
sns.pairplot(data.drop('country', axis=1), diag_kind='kde')
plt.suptitle('Pairplot con KDE en la diagonal', y=1.02) # Agregar un título
plt.tight_layout()
plt.show()

In [ ]:
# Antes de manipular los datos, guardamos una copia de los datos originales.
data_help = data.copy()
data_help.head()

In [ ]:
# Como podemos ver, hay varios outliers en los datos.

# Teniendo en cuenta que necesitamos identificar a los países rezagados según factores socioeconómicos y de salud,
# limitaremos (cap) los outliers a valores acordes para el análisis.

percentiles = data_help['child_mort'].quantile([0.05,0.95]).values
data_help['child_mort'][data_help['child_mort'] <= percentiles[0]] = percentiles[0]
data_help['child_mort'][data_help['child_mort'] >= percentiles[1]] = percentiles[1]

percentiles = data_help['health'].quantile([0.05,0.95]).values
data_help['health'][data_help['health'] <= percentiles[0]] = percentiles[0]
data_help['health'][data_help['health'] >= percentiles[1]] = percentiles[1]

percentiles = data_help['life_expec'].quantile([0.05,0.95]).values
data_help['life_expec'][data_help['life_expec'] <= percentiles[0]] = percentiles[0]
data_help['life_expec'][data_help['life_expec'] >= percentiles[1]] = percentiles[1]

percentiles = data_help['total_fer'].quantile([0.05,0.95]).values
data_help['total_fer'][data_help['total_fer'] <= percentiles[0]] = percentiles[0]
data_help['total_fer'][data_help['total_fer'] >= percentiles[1]] = percentiles[1]

percentiles = data_help['income'].quantile([0.05,0.95]).values
data_help['income'][data_help['income'] <= percentiles[0]] = percentiles[0]
data_help['income'][data_help['income'] >= percentiles[1]] = percentiles[1]

percentiles = data_help['inflation'].quantile([0.05,0.95]).values
data_help['inflation'][data_help['inflation'] <= percentiles[0]] = percentiles[0]
data_help['inflation'][data_help['inflation'] >= percentiles[1]] = percentiles[1]

percentiles = data_help['gdpp'].quantile([0.05,0.95]).values
data_help['gdpp'][data_help['gdpp'] <= percentiles[0]] = percentiles[0]
data_help['gdpp'][data_help['gdpp'] >= percentiles[1]] = percentiles[1]

percentiles = data_help['imports'].quantile([0.05,0.95]).values
data_help['imports'][data_help['imports'] <= percentiles[0]] = percentiles[0]
data_help['imports'][data_help['imports'] >= percentiles[1]] = percentiles[1]

percentiles = data_help['exports'].quantile([0.05,0.95]).values
data_help['exports'][data_help['exports'] <= percentiles[0]] = percentiles[0]
data_help['exports'][data_help['exports'] >= percentiles[1]] = percentiles[1]

In [ ]:
#
data_help.shape

In [ ]:
#
fig, axs = plt.subplots(3,3, figsize = (15,7.5))

plt1 = sns.boxplot(data_help['child_mort'], ax = axs[0,0])
plt2 = sns.boxplot(data_help['health'], ax = axs[0,1])
plt3 = sns.boxplot(data_help['life_expec'], ax = axs[0,2])
plt4 = sns.boxplot(data_help['total_fer'], ax = axs[1,0])
plt5 = sns.boxplot(data_help['income'], ax = axs[1,1])
plt6 = sns.boxplot(data_help['inflation'], ax = axs[1,2])
plt7 = sns.boxplot(data_help['gdpp'], ax = axs[2,0])
plt8 = sns.boxplot(data_help['imports'], ax = axs[2,1])
plt9 = sns.boxplot(data_help['exports'], ax = axs[2,2])

plt.tight_layout()

In [ ]:
# Generar un pairplot con KDE en la diagonal y scatter plots para los demás pares
sns.pairplot(data_help.drop('country', axis=1), diag_kind='kde')
plt.suptitle('Pairplot con KDE en la diagonal', y=1.02) # Agregar un título
plt.tight_layout()
plt.show()

### Escalando los datos

In [ ]:
# Crear un objeto de escalamiento
scaler = StandardScaler()

# Crear una lista de las variables que se deben escalar
varlist = ['child_mort', 'exports', 'health', 'imports', 'income', 'inflation', 'life_expec', 'total_fer', 'gdpp']
# Escalar estas variables usando 'fit_transform'
data_help[varlist] = scaler.fit_transform(data_help[varlist])

## PCA sobre los datos

In [ ]:
#
pca = PCA(svd_solver='randomized', random_state=42)

In [ ]:
# Poniendo las variables predictoras en X
X = data_help.drop(['country'],axis=1)

# Poniendo la variable de respuesta en y (los países como etiquetas)
y = data_help['country']

In [ ]:
#Ajustando el PCA sobre los datos
pca.fit(X)

#### Grafiquemos los componentes principales y tratemos de interpretarlos.
#### Graficaremos las características originales usando los primeros 2 componentes principales como ejes

In [ ]:
#
pca.components_

In [ ]:
#
colnames = list(X.columns)
pcs_df = pd.DataFrame({'PC1':pca.components_[0],'PC2':pca.components_[1], 'Feature':colnames})
pcs_df#.head()

In [ ]:
#
fig = plt.figure(figsize = (8,8))
plt.scatter(pcs_df.PC1, pcs_df.PC2)
plt.xlabel(f'Componente Principal 1 ({round(pca.explained_variance_ratio_[0]*100, 2)}%)')
plt.ylabel(f'Componente Principal 2 ({round(pca.explained_variance_ratio_[1]*100, 2)}%)')
for i, txt in enumerate(pcs_df.Feature):
    plt.annotate(txt, (pcs_df.PC1[i],pcs_df.PC2[i]))
plt.tight_layout()
plt.show()

#### Observando el scree plot para evaluar el número de componentes principales necesarios


In [ ]:
#
pca.explained_variance_ratio_

In [ ]:
#Generando el scree plot: varianza acumulada contra el número de componentes
fig = plt.figure(figsize = (12,8))
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel('número de componentes')
plt.ylabel('varianza explicada acumulada')
plt.savefig('pca_no')
plt.show()

## Clustering jerárquico

In [ ]:
#
df_pca = pd.DataFrame(pca.transform(X)[:, :4])

mergings = linkage(df_pca, method = "complete", metric='euclidean')
plt.figure(figsize=(20, 10)) # Tamaño de figura aumentado para un dendrograma más grande
dendrogram(mergings)
plt.ylabel('Distancia') # Etiquetando el eje y como 'Distancia'
plt.title('Dendrograma de clustering jerárquico') # Agregando un título para mayor claridad
plt.show()

In [ ]:
# Observando el dendrograma, se ve que cortarlo en n = 5 es lo más óptimo.

In [ ]:
#
clusterCut = pd.Series(cut_tree(mergings, n_clusters = 5).reshape(-1,))
df_pca_hc = pd.concat([df_pca, clusterCut], axis=1)
df_pca_hc.columns = ["PC1","PC2","PC3","PC4","ClusterID"]
df_pca_hc.head()


In [ ]:
#
pca_cluster_hc = pd.concat([data_help['country'],df_pca_hc], axis=1, join='outer', ignore_index=False, keys=None, levels=None, names=None, verify_integrity=False, sort=False, copy=True)
pca_cluster_hc.head()

In [ ]:
#
clustered_data_hc = pca_cluster_hc[['country','ClusterID']].merge(data, on = 'country')
clustered_data_hc.head()

In [ ]:
#
hc_clusters_child_mort = 	pd.DataFrame(clustered_data_hc.groupby(["ClusterID"]).child_mort.mean())
hc_clusters_exports = 	pd.DataFrame(clustered_data_hc.groupby(["ClusterID"]).exports.mean())
hc_clusters_health = 	pd.DataFrame(clustered_data_hc.groupby(["ClusterID"]).health.mean())
hc_clusters_imports = 	pd.DataFrame(clustered_data_hc.groupby(["ClusterID"]).imports.mean())
hc_clusters_income = 	pd.DataFrame(clustered_data_hc.groupby(["ClusterID"]).income.mean())
hc_clusters_inflation = 	pd.DataFrame(clustered_data_hc.groupby(["ClusterID"]).inflation.mean())
hc_clusters_life_expec = 	pd.DataFrame(clustered_data_hc.groupby(["ClusterID"]).life_expec.mean())
hc_clusters_total_fer = 	pd.DataFrame(clustered_data_hc.groupby(["ClusterID"]).total_fer.mean())
hc_clusters_gdpp = 	pd.DataFrame(clustered_data_hc.groupby(["ClusterID"]).gdpp.mean())

In [ ]:
#
df = pd.concat([pd.Series(list(range(0,5))), hc_clusters_child_mort,hc_clusters_exports, hc_clusters_health, hc_clusters_imports,
               hc_clusters_income, hc_clusters_inflation, hc_clusters_life_expec,hc_clusters_total_fer,hc_clusters_gdpp], axis=1)
df.columns = ["ClusterID", "child_mort_mean", "exports_mean", "health_mean", "imports_mean", "income_mean", "inflation_mean",
               "life_expec_mean", "total_fer_mean", "gdpp_mean"]
df

In [ ]:
#
fig, axs = plt.subplots(3,3,figsize = (15,15))

sns.barplot(x=df.ClusterID, y=df.child_mort_mean, ax = axs[0,0])
sns.barplot(x=df.ClusterID, y=df.exports_mean, ax = axs[0,1])
sns.barplot(x=df.ClusterID, y=df.health_mean, ax = axs[0,2])
sns.barplot(x=df.ClusterID, y=df.imports_mean, ax = axs[1,0])
sns.barplot(x=df.ClusterID, y=df.income_mean, ax = axs[1,1])
sns.barplot(x=df.ClusterID, y=df.life_expec_mean, ax = axs[1,2])
sns.barplot(x=df.ClusterID, y=df.inflation_mean, ax = axs[2,0])
sns.barplot(x=df.ClusterID, y=df.total_fer_mean, ax = axs[2,1])
sns.barplot(x=df.ClusterID, y=df.gdpp_mean, ax = axs[2,2])
plt.tight_layout()

In [ ]:
#
clustered_data_hc[clustered_data_hc.ClusterID == 0].country.values

### Recomendaciones
El cluster con ClusterID igual a 0 es el cluster de los países más rezagados.
